### Setup

In [1]:
import boto3
import requests
import feedparser
import pandas as pd

from datetime import datetime
from zoneinfo import ZoneInfo

from langgraph.graph import StateGraph, START, END
from typing import TypedDict

### Check if conversation is possible

In [2]:
MODEL_ID = "jp.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_runtime = boto3.Session(
    profile_name="bedrock-dev",
    region_name="ap-northeast-1"
).client("bedrock-runtime")

In [3]:
response = bedrock_runtime.converse(
    modelId=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": [
                {"text": "こんにちは。あなたのモデル名を教えてください。"}
            ]
        }
    ],
    inferenceConfig={
        "maxTokens": 200,
        "temperature": 0
    }
)

print(response["output"]["message"]["content"][0]["text"])

こんにちは。私はClaudeという名前のAIアシスタントです。Anthropicによって開発されました。

何かお手伝いできることがあれば、お気軽にお聞きください。


### Check using a weather API
#### Retrieve information using an API

In [4]:
import requests

# Retrieve location information for Fukui City
url = "https://geocoding-api.open-meteo.com/v1/search"

params = {
    "name": "Fukui",
    "count": 10,
    "language": "ja",
    "format": "json"
}

response = requests.get(url, params=params)
response.raise_for_status()

locations = response.json()["results"]


fukui = next(
    (
        location for location in locations
        if location.get("country_code") == "JP"
        and location.get("name") == "福井市"
    ),
    None
)

if fukui is None:
    raise ValueError("日本の福井市が見つかりませんでした")

latitude = fukui["latitude"]
longitude = fukui["longitude"]

print("都市:", fukui["name"])
print("緯度:", latitude)
print("経度:", longitude)

都市: 福井市
緯度: 36.06443
経度: 136.22256


#### Check what kind of information is included.

In [5]:
# Open-Meteo weather api

weather_url = "https://api.open-meteo.com/v1/forecast"

weather_params = {
    "latitude": latitude,
    "longitude": longitude,
    "current": [
        "temperature_2m",
        "apparent_temperature",
        "precipitation",
        "weather_code",
        "wind_speed_10m"
    ],
    "daily": [
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_probability_max",
        "weather_code"
    ],
    "timezone": "Asia/Tokyo",
    "forecast_days": 1
}

response = requests.get(
    weather_url,
    params=weather_params,
    timeout=30
)

response.raise_for_status()

weather_data = response.json()

print(weather_data)

{'latitude': 36.05, 'longitude': 136.25, 'generationtime_ms': 0.13208389282226562, 'utc_offset_seconds': 32400, 'timezone': 'Asia/Tokyo', 'timezone_abbreviation': 'GMT+9', 'elevation': 18.0, 'current_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'apparent_temperature': '°C', 'precipitation': 'mm', 'weather_code': 'wmo code', 'wind_speed_10m': 'km/h'}, 'current': {'time': '2026-11-22T12:45', 'interval': 900, 'temperature_2m': 9.1, 'apparent_temperature': 7.0, 'precipitation': 0.0, 'weather_code': 0, 'wind_speed_10m': 5.4}, 'daily_units': {'time': 'iso8601', 'temperature_2m_max': '°C', 'temperature_2m_min': '°C', 'precipitation_probability_max': '%', 'weather_code': 'wmo code'}, 'daily': {'time': ['2026-11-22'], 'temperature_2m_max': [9.5], 'temperature_2m_min': [3.0], 'precipitation_probability_max': [100], 'weather_code': [63]}}


#### Organize the necessary information.

In [6]:
current = weather_data["current"]
daily = weather_data["daily"]

weather_report = {
    "city": "福井市",
    "date": daily["time"][0],
    "temperature": current["temperature_2m"],
    "feels_like": current["apparent_temperature"],
    "wind_speed": current["wind_speed_10m"],
    "precipitation": current["precipitation"],
    "max_temperature": daily["temperature_2m_max"][0],
    "min_temperature": daily["temperature_2m_min"][0],
    "precipitation_probability": daily[
        "precipitation_probability_max"
    ][0],
    "weather_code": current["weather_code"]
}

print(weather_report)

{'city': '福井市', 'date': '2026-11-22', 'temperature': 9.1, 'feels_like': 7.0, 'wind_speed': 5.4, 'precipitation': 0.0, 'max_temperature': 9.5, 'min_temperature': 3.0, 'precipitation_probability': 100, 'weather_code': 0}


#### Quantify the weather by assigning it a score, making it easier to understand.

In [7]:
# Weather Information
weather_summary = weather_report

# Weather Code Conversion
weather_code_map = {
    0: "快晴",
    1: "晴れ",
    2: "一部曇り",
    3: "曇り",
    51: "弱い霧雨",
    53: "中程度の霧雨",
    61: "弱い雨",
    63: "中程度の雨",
    65: "強い雨"
}

weather_summary["weather"] = weather_code_map.get(
    weather_summary["weather_code"],
    "その他"
)

print(weather_summary)


{'city': '福井市', 'date': '2026-11-22', 'temperature': 9.1, 'feels_like': 7.0, 'wind_speed': 5.4, 'precipitation': 0.0, 'max_temperature': 9.5, 'min_temperature': 3.0, 'precipitation_probability': 100, 'weather_code': 0, 'weather': '快晴'}


### Using a weather API with LangGraph
#### Setup

In [8]:
import os
import boto3

os.environ["AWS_PROFILE"] = "bedrock-dev"
AWS_REGION = "ap-northeast-1"

session = boto3.Session()

In [9]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model="jp.anthropic.claude-haiku-4-5-20251001-v1:0",
    region_name=AWS_REGION,
    credentials_profile_name="bedrock-dev",
    temperature=0
)

#### Build an AI agent that feeds weather data into an LLM to automatically generate weather analysis reports.

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# Data Handled by Agents
class WeatherState(TypedDict):
    weather: dict
    analysis: str

# Weather Analysis Node
def analyze_weather(state: WeatherState):

    prompt = f"""
    あなたは天気分析の専門家です。

    以下の天気情報を分析してください。

    天気情報:
    {state["weather"]}

    次の3点を回答してください。
    1. 現在の天気情報
    2. 外出時の注意点
    3. 具体的な行動提案
    """

    response = llm.invoke(prompt)

    return {
        "analysis": response.content
    }

# Building with LangGraph
builder = StateGraph(WeatherState)

builder.add_node("analyze_weather", analyze_weather)

builder.add_edge(START, "analyze_weather")
builder.add_edge("analyze_weather", END)

weather_agent = builder.compile()

#### Connect the aforementioned API retrieval process to the agent execution process.

In [11]:
# Run the weather analysis agent
result = weather_agent.invoke({
    "weather": weather_summary,
    "analysis": ""
})

# Display analysis results
print(result["analysis"])
print()
print(result)

# 福井市の天気分析レポート（2026年11月22日）

## 1. 現在の天気情報

| 項目 | 詳細 |
|------|------|
| **天気** | 快晴 ☀️ |
| **気温** | 9.1℃（体感気温：7.0℃） |
| **気温範囲** | 最高9.5℃ / 最低3.0℃ |
| **風速** | 5.4m/s（やや強い風） |
| **降水確率** | 100%※ |
| **降水量** | 0.0mm |

**注記**：降水確率100%と降水量0.0mmは矛盾していますが、現在は雨が降っていない状態です。

---

## 2. 外出時の注意点

⚠️ **気温が低い**
- 体感気温が7℃と実気温より2℃低く感じられます
- 朝方は3℃まで冷え込む可能性があります

💨 **風が強い**
- 5.4m/sの風は日常生活に影響を与える強さです
- 髪や帽子が飛ばされやすい状態です

☀️ **紫外線対策**
- 快晴のため紫外線が強い可能性があります

---

## 3. 具体的な行動提案

✅ **服装**
- 厚手のジャケットやコート必須
- マフラーや手袋の着用を推奨
- 帽子は風で飛ばされないようしっかり固定

✅ **外出活動**
- 快晴なので屋外活動に最適
- ハイキングや散歩は良好な条件
- 洗濯物の干し物に適した日

✅ **その他**
- 風が強いため、自転車や原付は注意
- 看板や植木鉢など飛ばされやすい物の確認
- 日焼け止めの使用を検討

{'weather': {'city': '福井市', 'date': '2026-11-22', 'temperature': 9.1, 'feels_like': 7.0, 'wind_speed': 5.4, 'precipitation': 0.0, 'max_temperature': 9.5, 'min_temperature': 3.0, 'precipitation_probability': 100, 'weather_code': 0, 'weather': '快晴'}, 'analysis': '# 福井市の天気分析レポート（2026年11月22日）\n\n## 1. 現在の天気情報\n\n| 項目 | 詳細 |\n|------|------|\n|

### Fitness Agent
#### Definition of FitnessState

In [12]:
class FitnessState(TypedDict):

    goal: str
    available_minutes: int
    fatigue_level: list[str]
    symptoms: list[str]

    analysis: str
    safety_result: str
    workout_plan: str

#### Create initial state

In [13]:
initial_state: FitnessState = {
    "goal": "持久力向上",
    "available_minutes": 30,
    "fatigue_level": 2,
    "recent_workouts": [
        "昨日:5kmランニング",
        "一昨日:休息",
        "3日前：筋力トレーニング"
    ],
    "symptoms": [],

    "analysis": "",
    "safety_result": "",
    "workout_plan": ""
}

print(initial_state)

{'goal': '持久力向上', 'available_minutes': 30, 'fatigue_level': 2, 'recent_workouts': ['昨日:5kmランニング', '一昨日:休息', '3日前：筋力トレーニング'], 'symptoms': [], 'analysis': '', 'safety_result': '', 'workout_plan': ''}


#### Creating common functions

In [14]:
FITNESS_SYSTEM_PROMPT = """
    あなたはFitness Agentです。

    ユーザーの運動目的や体調を考慮して、
    適切な運動メニューを提案してください。

    【基本ルール】
    - 運動目的を優先する
    - 指定された時間内に収める
    - 疲労度に応じて運動強度を調整する
    - 痛みや体調不良がある場合は安全を優先する
    - 不明な情報を勝手に補完しない

    【出力ルール】
    - JSON形式のみで出力する
    - Markdownのコードブロックは使用しない
    - 指定されたJSON構造を維持する
"""

def invoke_llm(
    system_prompt: str,
    user_prompt: str,
) -> str:

    response = llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ])

    return response.content